---
## 0. Cài đặt & Kiểm tra môi trường

In [ ]:
# Cài đặt các thư viện cần thiết (bỏ comment nếu chưa cài)
# !pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
# !pip install timm ultralytics albumentations
# !pip install matplotlib seaborn scikit-learn ipywidgets

In [ ]:
import os, json, time, warnings
from pathlib import Path
from copy import deepcopy

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageEnhance, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

import timm
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {name} ({mem:.1f} GB VRAM)')
else:
    DEVICE = torch.device('cpu')
    print('Khong co GPU — chay tren CPU')

print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')

---
## 1. Cau hinh — Chinh tai day!

> Moi thong so tap trung trong `CFG`. Khong can sua code o cac cell phia sau.

In [ ]:
CFG = {
    # Duong dan
    'data_dir' : 'dataset',
    'out_dir'  : 'models_output',

    # Model
    # Cac lua chon:
    #   'tf_efficientnetv2_s.in21ft1k'          mac dinh, can bang tot
    #   'mobilenetv3_large_100.ra_in1k'          nhe nhat, deploy CPU
    #   'resnet50.a1_in1k'                        baseline kinh dien
    #   'convnext_tiny.fb_in22k_ft_in1k'         modern CNN
    #   'vit_small_patch16_224.augreg_in1k'      Vision Transformer
    #   'swin_tiny_patch4_window7_224.ms_in1k'   Swin Transformer
    'model_name'  : 'tf_efficientnetv2_s.in21ft1k',
    'img_size'    : 224,
    'checkpoint'  : '',   # duong dan .pt de fine-tune tu checkpoint, de '' neu train moi

    # Training
    'epochs'       : 50,
    'batch_size'   : 16,
    'warmup_ratio' : 0.1,
    'num_workers'  : 4,

    # Loss Function
    # Chon 1 trong 4:
    #   'cross_entropy'    chuan, dung khi du lieu balanced
    #   'label_smoothing'  giam overfit, tot cho dataset nho
    #   'focal'            tot nhat khi du lieu mat can bang nang
    #   'asymmetric'       focal cai tien, tot cho multi-label
    'loss'            : 'focal',
    'label_smoothing' : 0.1,
    'focal_gamma'     : 2.0,
    'focal_alpha'     : 0.25,

    # Optimizer
    # Chon 1 trong 4:
    #   'adamw'    mac dinh, on dinh, hoi tu nhanh
    #   'sgd'      tot cho finetune sau, can lr cao hon
    #   'rmsprop'  tot cho du lieu temporal/spectral
    #   'lion'     moi nhat (2023), tiet kiem memory
    'optimizer'    : 'adamw',
    'lr_head'      : 1e-3,
    'lr_full'      : 1e-4,
    'weight_decay' : 1e-4,
    'momentum'     : 0.9,


    # Early Stopping
    'early_stop'         : True,
    'early_stop_patience': 10,
    'early_stop_delta'   : 0.001,
    'early_stop_metric'  : 'val_acc',   # 'val_acc' hoac 'val_loss'

    # Augmentation
    'aug_brightness' : 0.2,
    'aug_contrast'   : 0.2,
    'aug_translate'  : 0.05,
    'aug_blur_sigma' : (0.1, 0.5),
}

OUT_DIR  = Path(CFG['out_dir'])
DATA_DIR = Path(CFG['data_dir'])
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Cau hinh hien tai:')
groups = [
    ('Model'    , ['model_name','img_size','checkpoint']),
    ('Training' , ['epochs','batch_size','warmup_ratio']),
    ('Loss'     , ['loss','label_smoothing','focal_gamma','focal_alpha']),
    ('Optimizer', ['optimizer','lr_head','lr_full','weight_decay']),
    ('EarlyStop', ['early_stop','early_stop_patience','early_stop_delta','early_stop_metric']),
]
for group, keys in groups:
    print(f'  [{group}]')
    for k in keys:
        print(f'    {k:<25}: {CFG[k]}')

---
## 2. Tien xu ly & Kham pha du lieu (EDA)

### 2a. Xem anh mau

In [ ]:
def show_sample_images(data_dir, split='train', n_per_class=4):
    split_dir = Path(data_dir) / split
    if not split_dir.exists():
        print(f'Khong tim thay: {split_dir}')
        return []
    classes = sorted([d.name for d in split_dir.iterdir() if d.is_dir()])
    if not classes:
        return []
    fig, axes = plt.subplots(len(classes), n_per_class,
                              figsize=(n_per_class*3, len(classes)*3))
    if len(classes) == 1:
        axes = axes[np.newaxis, :]
    for r, cls in enumerate(classes):
        imgs = [p for p in sorted((split_dir/cls).glob('*'))
                if p.suffix.lower() in ('.png','.jpg','.jpeg','.bmp')]
        for c, p in enumerate(imgs[:n_per_class]):
            img = Image.open(p).convert('RGB')
            axes[r,c].imshow(img)
            axes[r,c].set_title(f'{cls}\n{img.size[0]}x{img.size[1]}', fontsize=9)
            axes[r,c].axis('off')
        for c in range(len(imgs[:n_per_class]), n_per_class):
            axes[r,c].axis('off')
    fig.suptitle(f'Anh mau - {split}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR/f'sample_{split}.png', dpi=120, bbox_inches='tight')
    plt.show()
    return classes

CLASS_NAMES = show_sample_images(DATA_DIR, 'train', 4)

### 2b. Phan phoi du lieu

In [ ]:
def analyze_dataset(data_dir):
    splits = ['train','val','test']
    stats  = {}
    for s in splits:
        d = Path(data_dir)/s
        if not d.exists(): continue
        stats[s] = {c.name: len([f for f in c.iterdir()
                    if f.suffix.lower() in ('.png','.jpg','.jpeg','.bmp')])
                    for c in sorted(d.iterdir()) if c.is_dir()}
    if not stats: return stats
    all_cls = sorted({c for s in stats.values() for c in s})
    x, w = np.arange(len(all_cls)), 0.25
    colors = ['#4C72B0','#55A868','#C44E52']
    fig, ax = plt.subplots(figsize=(max(8, len(all_cls)*1.5), 5))
    for i,(s,col) in enumerate(zip(splits,colors)):
        if s not in stats: continue
        vals = [stats[s].get(c,0) for c in all_cls]
        bars = ax.bar(x+i*w, vals, w, label=s, color=col, alpha=0.85)
        for bar,v in zip(bars,vals):
            if v>0: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                            str(v), ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x+w); ax.set_xticklabels(all_cls, fontsize=11)
    ax.set_ylabel('So anh'); ax.legend()
    ax.set_title('Phan phoi du lieu', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR/'data_distribution.png', dpi=120)
    plt.show()
    print('\nThong ke:')
    for cls in all_cls:
        row = f'  {cls:<20}' + ''.join(f'{stats[s].get(cls,0):>8}' for s in splits if s in stats)
        print(row)
    return stats

dataset_stats = analyze_dataset(DATA_DIR)

### 2c. Preview mau sac & Augmentation

In [ ]:
def _equalize(img):
    from PIL import ImageOps
    return ImageOps.equalize(img)

def _invert(img):
    from PIL import ImageOps
    return ImageOps.invert(img)

def _gamma(img, g):
    arr = np.array(img).astype(np.float32)/255.0
    return Image.fromarray((np.power(arr,g)*255).clip(0,255).astype(np.uint8))

sample_img = None
train_p = DATA_DIR/'train'
if train_p.exists():
    for cls_dir in sorted(train_p.iterdir()):
        if cls_dir.is_dir():
            for f in cls_dir.iterdir():
                if f.suffix.lower() in ('.png','.jpg','.jpeg'):
                    sample_img = Image.open(f).convert('RGB')
                    print(f'Anh mau: {f}  |  {sample_img.size}')
                    break
        if sample_img: break

if sample_img is None:
    print('Dung anh demo (chua co dataset)')
    arr = np.random.randint(0,80,(128,256,3),dtype=np.uint8)
    arr[40:50,80:100]=[200,80,50]; arr[70:80,140:170]=[80,200,50]
    sample_img = Image.fromarray(arr)

In [ ]:
W    = CFG['img_size']
base = sample_img.resize((W,W))

adjustments = [
    ('Original',         base),
    ('Brightness +0.5',  ImageEnhance.Brightness(base).enhance(1.5)),
    ('Brightness -0.5',  ImageEnhance.Brightness(base).enhance(0.5)),
    ('Contrast x2',      ImageEnhance.Contrast(base).enhance(2.0)),
    ('Contrast x0.5',    ImageEnhance.Contrast(base).enhance(0.5)),
    ('Sharpness x3',     ImageEnhance.Sharpness(base).enhance(3.0)),
    ('Blur s=2',         base.filter(ImageFilter.GaussianBlur(radius=2))),
    ('Grayscale',        base.convert('L').convert('RGB')),
    ('Equalize',         _equalize(base)),
    ('Invert',           _invert(base)),
    ('Gamma 0.5 (toi)',  _gamma(base, 0.5)),
    ('Gamma 2.0 (sang)', _gamma(base, 2.0)),
]

cols = 4; rows = -(-len(adjustments)//cols)
fig, axes = plt.subplots(rows, cols, figsize=(18,10))
for ax,(title,im) in zip(axes.flatten(), adjustments):
    ax.imshow(im); ax.set_title(title, fontsize=10, fontweight='bold'); ax.axis('off')
for ax in axes.flatten()[len(adjustments):]:
    ax.axis('off')
fig.suptitle('Dieu chinh mau sac & bo loc', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR/'color_adjustments.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
aug_preview = transforms.Compose([
    transforms.Resize((W,W)),
    transforms.RandomAffine(degrees=0, translate=(CFG['aug_translate'],CFG['aug_translate'])),
    transforms.ColorJitter(brightness=CFG['aug_brightness'], contrast=CFG['aug_contrast']),
    transforms.GaussianBlur(kernel_size=3, sigma=CFG['aug_blur_sigma']),
])

n = 8
fig, axes = plt.subplots(2, (n+2)//2, figsize=(16,6))
axes = axes.flatten()
axes[0].imshow(base); axes[0].set_title('Original', fontweight='bold', color='green'); axes[0].axis('off')
for i in range(1, n+1):
    axes[i].imshow(aug_preview(base)); axes[i].set_title(f'Aug #{i}', fontsize=9); axes[i].axis('off')
for ax in axes[n+1:]: ax.axis('off')
fig.suptitle('Preview Augmentation (training transform)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR/'augmentation_preview.png', dpi=120)
plt.show()

---
## 3. Dataloader & Transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((W,W)),
    transforms.RandomAffine(degrees=0, translate=(CFG['aug_translate'],CFG['aug_translate'])),
    transforms.ColorJitter(brightness=CFG['aug_brightness'], contrast=CFG['aug_contrast']),
    transforms.GaussianBlur(kernel_size=3, sigma=CFG['aug_blur_sigma']),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((W,W)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

train_ds = datasets.ImageFolder(DATA_DIR/'train', transform=train_transform)
val_ds   = datasets.ImageFolder(DATA_DIR/'val',   transform=val_transform)
test_ds  = datasets.ImageFolder(DATA_DIR/'test',  transform=val_transform)

CLASS_NAMES = train_ds.classes
NUM_CLASSES = len(CLASS_NAMES)

class_counts  = np.bincount([y for _,y in train_ds.samples])
class_weights = torch.FloatTensor(1.0/class_counts).to(DEVICE)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                           num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False,
                           num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=CFG['batch_size'], shuffle=False,
                           num_workers=2, pin_memory=True)

print(f'Lop phan loai  : {CLASS_NAMES}')
print(f'Train          : {len(train_ds)} anh  ({len(train_loader)} batches)')
print(f'Val            : {len(val_ds)} anh')
print(f'Test           : {len(test_ds)} anh')
print(f'Class weights  : {class_weights.cpu().numpy().round(3)}')

---
## 4. Khoi tao Model

In [ ]:
def build_model(model_name, num_classes, pretrained=True):
    available = set(timm.list_models(pretrained=pretrained))
    if model_name in available:
        chosen = model_name
    else:
        base    = model_name.split('.')[0]
        matches = timm.list_models(f'*{base}*', pretrained=True)
        chosen  = sorted(matches)[0] if matches else 'efficientnet_b3.ra2_in1k'
        print(f'  "{model_name}" -> dung "{chosen}"')
    print(f'Model: {chosen}')
    m     = timm.create_model(chosen, pretrained=pretrained, num_classes=num_classes)
    total = sum(p.numel() for p in m.parameters())
    print(f'Params: {total/1e6:.2f}M')
    return m, chosen


model, CHOSEN_MODEL = build_model(CFG['model_name'], NUM_CLASSES)

# Load checkpoint neu co
if CFG['checkpoint'] and Path(CFG['checkpoint']).exists():
    state = torch.load(CFG['checkpoint'], map_location='cpu')
    try:
        model.load_state_dict(state)
        print(f'Loaded checkpoint (full): {CFG["checkpoint"]}')
    except RuntimeError:
        HEAD_KW = ('classifier','head','fc')
        state_f = {k:v for k,v in state.items() if not any(h in k for h in HEAD_KW)}
        model.load_state_dict(state_f, strict=False)
        print(f'Loaded checkpoint (backbone only, head reset): {CFG["checkpoint"]}')

model = model.to(DEVICE)

# Phase 1: freeze backbone
HEAD_KEYWORDS = ('classifier','head','fc')
for name, param in model.named_parameters():
    if not any(k in name for k in HEAD_KEYWORDS):
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Phase 1 — train head only: {trainable/1e3:.1f}K params')

---
## 5. Loss Function


In [ ]:
# FOCAL LOSS
# Tap trung vao mau kho phan loai. gamma cao -> tap trung nhieu hon.
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, weight=None, reduction='mean'):
        super().__init__()
        self.gamma     = gamma
        self.alpha     = alpha
        self.weight    = weight
        self.reduction = reduction

    def forward(self, logits, targets):
        ce    = F.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt    = torch.exp(-ce)
        focal = self.alpha * (1 - pt) ** self.gamma * ce
        return focal.mean() if self.reduction == 'mean' else focal.sum()

    def __repr__(self):
        return f'FocalLoss(gamma={self.gamma}, alpha={self.alpha})'


# ASYMMETRIC LOSS
# Phat khac nhau cho false-positive va false-negative.
# gamma_neg > gamma_pos -> giam dong gop cua negative de.
class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_neg=4, gamma_pos=1, clip=0.05, weight=None):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip
        self.weight    = weight

    def forward(self, logits, targets):
        prob   = torch.softmax(logits, dim=1)
        oh     = F.one_hot(targets, num_classes=logits.size(1)).float()
        prob_m = (prob - self.clip).clamp(min=1e-8)
        ce_pos = oh     * torch.log(prob.clamp(min=1e-8))
        ce_neg = (1-oh) * torch.log((1-prob_m).clamp(min=1e-8))
        loss   = -(  oh*(1-prob)**self.gamma_pos * ce_pos
                  + (1-oh)*prob_m**self.gamma_neg * ce_neg )
        if self.weight is not None:
            loss = loss * self.weight.unsqueeze(0)
        return loss.sum(dim=1).mean()

    def __repr__(self):
        return f'AsymmetricLoss(gamma_neg={self.gamma_neg}, gamma_pos={self.gamma_pos})'


# FACTORY
def build_loss(cfg, class_weights):
    choice = cfg['loss']
    if choice == 'cross_entropy':
        return nn.CrossEntropyLoss(weight=class_weights)
    elif choice == 'label_smoothing':
        return nn.CrossEntropyLoss(weight=class_weights,
                                    label_smoothing=cfg['label_smoothing'])
    elif choice == 'focal':
        return FocalLoss(gamma=cfg['focal_gamma'], alpha=cfg['focal_alpha'],
                          weight=class_weights)
    elif choice == 'asymmetric':
        return AsymmetricLoss(weight=class_weights)
    else:
        raise ValueError(f'Loss khong hop le: {choice}')


criterion = build_loss(CFG, class_weights)
print(f'Loss: {criterion}')

---
## 6. Optimizer


In [ ]:
def build_optimizer(cfg, params, lr):
    choice = cfg['optimizer']
    wd     = cfg['weight_decay']
    if choice == 'adamw':
        return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    elif choice == 'sgd':
        return torch.optim.SGD(params, lr=lr, momentum=cfg['momentum'],
                                weight_decay=wd, nesterov=True)
    elif choice == 'rmsprop':
        return torch.optim.RMSprop(params, lr=lr, momentum=cfg['momentum'],
                                    weight_decay=wd)
    elif choice == 'lion':
        try:
            from lion_pytorch import Lion
            return Lion(params, lr=lr/10, weight_decay=wd)
        except ImportError:
            print('lion_pytorch chua cai (pip install lion-pytorch) -> fallback AdamW')
            return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    else:
        raise ValueError(f'Optimizer khong hop le: {choice}')



# Phase 1 optimizer (head only)
head_params = [p for n,p in model.named_parameters()
               if any(k in n for k in HEAD_KEYWORDS) and p.requires_grad]
if not head_params:
    print('Khong tim duoc head -> unfreeze toan bo')
    for p in model.parameters(): p.requires_grad = True
    head_params = list(model.parameters())

optimizer = build_optimizer(CFG, head_params, CFG['lr_head'])
print(f'Optimizer : {CFG["optimizer"].upper()}  (lr_head={CFG["lr_head"]})')


---
## 7. Early Stopping


In [ ]:
class EarlyStopping:
    """
    Theo doi val_acc hoac val_loss va dung training khi khong con cai thien.

    Cach dung:
        es = EarlyStopping(patience=10, delta=0.001, metric='val_acc')
        for epoch in range(epochs):
            ...train...
            if es(val_acc, model, epoch):
                break
    """
    def __init__(self, patience=10, delta=0.001, metric='val_acc',
                 checkpoint_path='best_classifier.pt', verbose=True):
        self.patience   = patience
        self.delta      = delta
        self.metric     = metric
        self.path       = checkpoint_path
        self.verbose    = verbose
        self.counter    = 0
        self.best_score = None
        self.best_epoch = 0
        self.stopped    = False
        self._maximize  = (metric == 'val_acc')

    def _is_better(self, score):
        if self.best_score is None:
            return True
        return (score > self.best_score + self.delta) if self._maximize \
               else (score < self.best_score - self.delta)

    def __call__(self, score, model, epoch):
        """Tra ve True neu nen dung training."""
        if self._is_better(score):
            if self.verbose and self.best_score is not None:
                d = 'up' if self._maximize else 'dn'
                print(f'         [{d}] {self.metric}: '
                      f'{self.best_score:.4f} -> {score:.4f}  [saved]')
            self.best_score = score
            self.best_epoch = epoch
            self.counter    = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            if self.verbose:
                print(f'         No improvement ({self.counter}/{self.patience})')
            if self.counter >= self.patience:
                self.stopped = True
                print(f'\n  Early stopping! '
                      f'Best {self.metric}={self.best_score:.4f} '
                      f'tai epoch {self.best_epoch+1}')
                return True
        return False

    def load_best(self, model, device):
        model.load_state_dict(torch.load(self.path, map_location=device))
        print(f'Loaded best model (epoch {self.best_epoch+1}, '
              f'{self.metric}={self.best_score:.4f})')
        return model


early_stopper = EarlyStopping(
    patience        = CFG['early_stop_patience'],
    delta           = CFG['early_stop_delta'],
    metric          = CFG['early_stop_metric'],
    checkpoint_path = str(OUT_DIR/'best_classifier.pt'),
    verbose         = True,
) if CFG['early_stop'] else None

if early_stopper:
    print(f'Early Stopping: patience={CFG["early_stop_patience"]}  '
          f'delta={CFG["early_stop_delta"]}  '
          f'metric={CFG["early_stop_metric"]}')
else:
    print('Early Stopping: TAT — chay het toan bo epochs')

---
## 8. Training Loop

In [ ]:
history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[]}
best_val_acc  = 0.0
WARMUP_EPOCHS = max(1, int(CFG['epochs'] * CFG['warmup_ratio']))

print(f'\n{"="*60}')
print(f'Training  : {CFG["epochs"]} epochs max | device={DEVICE}')
print(f'Phase 1   : epochs 1 -> {WARMUP_EPOCHS}  (head only)')
print(f'Phase 2   : epochs {WARMUP_EPOCHS+1} -> {CFG["epochs"]}  (full finetune)')
print(f'Loss      : {criterion}')
print(f'Optimizer : {CFG["optimizer"]}')
es_str = f'patience={CFG["early_stop_patience"]}' if CFG['early_stop'] else 'OFF'
print(f'EarlyStop : {es_str}')
print(f'{"="*60}\n')

for epoch in range(CFG['epochs']):

    # Phase 2: unfreeze backbone
    if epoch == WARMUP_EPOCHS:
        print(f'\n[Epoch {epoch+1}] Phase 2: unfreeze toan bo backbone')
        for p in model.parameters():
            p.requires_grad = True
        optimizer     = build_optimizer(CFG, model.parameters(), CFG['lr_full'])

    # Train
    model.train()
    tr_loss, tr_correct, tr_total = 0.0, 0, 0
    t0 = time.time()

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        out  = model(imgs)
        loss = criterion(out, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        tr_loss    += loss.item() * imgs.size(0)
        tr_correct += (out.argmax(1) == labels).sum().item()
        tr_total   += imgs.size(0)

    # Validate
    model.eval()
    va_loss, va_correct, va_total = 0.0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            out  = model(imgs)
            loss = criterion(out, labels)
            va_loss    += loss.item() * imgs.size(0)
            va_correct += (out.argmax(1) == labels).sum().item()
            va_total   += imgs.size(0)
    current_lr = optimizer.param_groups[0]['lr']

    t_loss = tr_loss/tr_total;  t_acc = tr_correct/tr_total*100
    v_loss = va_loss/va_total;  v_acc = va_correct/va_total*100

    history['train_loss'].append(t_loss); history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc);   history['val_acc'].append(v_acc)

    elapsed = time.time() - t0
    print(f'  [{epoch+1:3d}/{CFG["epochs"]}]  '
          f'train {t_acc:.1f}%/{t_loss:.4f}   '
          f'val {v_acc:.1f}%/{v_loss:.4f}   ' ({elapsed:.1f}s)')

    # Early Stopping
    es_score = v_acc if CFG['early_stop_metric'] == 'val_acc' else v_loss
    if early_stopper:
        if early_stopper(es_score, model, epoch):
            break
    else:
        if v_acc > best_val_acc:
            best_val_acc = v_acc
            torch.save(model.state_dict(), OUT_DIR/'best_classifier.pt')

# Load best model
if early_stopper:
    model        = early_stopper.load_best(model, DEVICE)
    best_val_acc = early_stopper.best_score
    stopped_epoch = early_stopper.best_epoch + 1
else:
    model.load_state_dict(torch.load(OUT_DIR/'best_classifier.pt', map_location=DEVICE))
    stopped_epoch = len(history['val_acc'])

print(f'\nTraining xong! Best val_acc={best_val_acc:.2f}%  (epoch {stopped_epoch})')

---
## 9. Danh gia & Visualize

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

curves = [
    (history['train_loss'], history['val_loss'], 'Loss'),
    (history['train_acc'],  history['val_acc'],  'Accuracy (%)'),
    (history['lr'],         None,                'Learning Rate'),
]
for ax, (y1, y2, title) in zip(axes, curves):
    ax.plot(y1, label='Train' if y2 else 'LR', linewidth=2)
    if y2: ax.plot(y2, label='Val', linewidth=2)
    ax.axvline(WARMUP_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='Phase 2')
    if early_stopper and early_stopper.stopped:
        ax.axvline(early_stopper.best_epoch, color='red', linestyle=':', alpha=0.7, label='Best')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(alpha=0.3)

fig.suptitle(f'{CHOSEN_MODEL}  |  loss={CFG["loss"]}  opt={CFG["optimizer"]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR/'training_curves.png', dpi=150)
plt.show()

In [ ]:
# Test set evaluation
model.eval()
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs  = imgs.to(DEVICE)
        out   = model(imgs)
        probs = torch.softmax(out, dim=1)
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

print('Classification Report (Test Set):')
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
# Confusion Matrix
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14,5))
sns.heatmap(cm,      annot=True, fmt='d',   cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2)
for ax, t in zip([ax1,ax2],['So luong','Ty le']):
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix ({t})', fontweight='bold')
plt.tight_layout()
plt.savefig(OUT_DIR/'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Anh phan loai sai
wrong = [i for i,(p,l) in enumerate(zip(all_preds,all_labels)) if p!=l]
if not wrong:
    print('Khong co anh nao sai tren test set!')
else:
    n    = min(8, len(wrong))
    MEAN = torch.tensor([0.485,0.456,0.406])
    STD  = torch.tensor([0.229,0.224,0.225])
    fig, axes = plt.subplots(2, max(n//2,1), figsize=(3*max(n//2,1), 7))
    for ax, idx in zip(axes.flatten(), wrong[:n]):
        t, tl = test_ds[idx]
        img = (t*STD[:,None,None]+MEAN[:,None,None]).permute(1,2,0).numpy().clip(0,1)
        ax.imshow(img)
        ax.set_title(f'True: {CLASS_NAMES[tl]}\nPred: {CLASS_NAMES[all_preds[idx]]}',
                     fontsize=9, color='red')
        ax.axis('off')
    fig.suptitle(f'Anh sai: {len(wrong)}/{len(all_labels)}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(OUT_DIR/'misclassified.png', dpi=120)
    plt.show()

---
## 10. Export ONNX

In [ ]:
model.eval()
dummy     = torch.randn(1,3,W,W).to(DEVICE)
onnx_path = OUT_DIR/'classifier.onnx'

torch.onnx.export(
    model, dummy, str(onnx_path),
    opset_version=17,
    input_names=['input'], output_names=['output'],
    dynamic_axes={'input':{0:'batch'},'output':{0:'batch'}},
)
print(f'ONNX: {onnx_path}  ({onnx_path.stat().st_size/1e6:.1f} MB)')

try:
    import onnxruntime as ort
    sess = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
    inp  = dummy.cpu().numpy()
    name = sess.get_inputs()[0].name
    sess.run(None, {name: inp})  # warmup
    times = []
    for _ in range(50):
        t = time.perf_counter()
        sess.run(None, {name: inp})
        times.append((time.perf_counter()-t)*1000)
    print(f'CPU inference: {np.mean(times):.1f} +/- {np.std(times):.1f} ms/anh')
except ImportError:
    print('pip install onnxruntime de verify')

meta = {
    'classes'     : CLASS_NAMES,
    'num_classes' : NUM_CLASSES,
    'model'       : CHOSEN_MODEL,
    'img_size'    : W,
    'loss'        : CFG['loss'],
    'optimizer'   : CFG['optimizer'],
    'best_val_acc': round(float(best_val_acc), 4),
    'best_epoch'  : int(stopped_epoch),
}
with open(OUT_DIR/'class_labels.json','w') as f:
    json.dump(meta, f, indent=2)

print(f'\n{"="*55}')
print(f'TONG KET')
print(f'Best val_acc : {best_val_acc:.2f}%  (epoch {stopped_epoch})')
print(f'Model        : {CHOSEN_MODEL}')
print(f'Loss         : {CFG["loss"]}  |  Optimizer: {CFG["optimizer"]}')
print(f'Files        : {OUT_DIR}/')
print(f'  best_classifier.pt  classifier.onnx  class_labels.json')
print(f'{"="*55}')
print(f'-> python step3_deploy.py --model {OUT_DIR}/classifier.onnx')

---
## 11. Benchmark nhieu model (Tuy chon)

In [ ]:
MODELS_TO_COMPARE = [
    'tf_efficientnetv2_s.in21ft1k',
    'mobilenetv3_large_100.ra_in1k',
    'resnet50.a1_in1k',
    'convnext_tiny.fb_in22k_ft_in1k',
    'vit_small_patch16_224.augreg_in1k',
]
QUICK_EPOCHS = 3

def quick_benchmark(model_names, num_classes, train_loader, val_loader, device, epochs=3):
    results = []
    for mname in model_names:
        print(f'\n> {mname}')
        try:
            m, chosen = build_model(mname, num_classes)
            m = m.to(device)
            for p in m.parameters(): p.requires_grad = False
            for p in list(m.children())[-1].parameters(): p.requires_grad = True
            opt  = torch.optim.AdamW(filter(lambda p: p.requires_grad, m.parameters()), lr=1e-3)
            crit = build_loss(CFG, class_weights)
            for _ in range(epochs):
                m.train()
                for imgs,lbls in train_loader:
                    imgs,lbls = imgs.to(device),lbls.to(device)
                    opt.zero_grad(); loss=crit(m(imgs),lbls); loss.backward(); opt.step()
            m.eval()
            correct,total = 0,0
            with torch.no_grad():
                for imgs,lbls in val_loader:
                    correct += (m(imgs.to(device)).argmax(1).cpu()==lbls).sum().item()
                    total   += lbls.size(0)
            val_acc = correct/total*100
            dummy2  = torch.randn(1,3,W,W)
            m_cpu   = m.cpu().eval()
            with torch.no_grad():
                t0=time.perf_counter()
                for _ in range(20): m_cpu(dummy2)
                inf_ms = (time.perf_counter()-t0)/20*1000
            params_m = sum(p.numel() for p in m.parameters())/1e6
            results.append({'model':chosen,'val_acc':val_acc,'inf_ms':inf_ms,'params_M':params_m})
            print(f'  val={val_acc:.1f}%  inf={inf_ms:.1f}ms  params={params_m:.1f}M')
        except Exception as e:
            print(f'  Error: {e}')
    return results

# Bo comment de chay:
# bench = quick_benchmark(MODELS_TO_COMPARE, NUM_CLASSES, train_loader, val_loader, DEVICE, QUICK_EPOCHS)
print('Bo comment dong cuoi de chay benchmark.')

In [ ]:
def plot_benchmark(results):
    if not results: return
    names  = [r['model'].split('.')[0] for r in results]
    accs   = [r['val_acc']  for r in results]
    speeds = [r['inf_ms']   for r in results]
    params = [r['params_M'] for r in results]
    fig, axes = plt.subplots(1, 3, figsize=(16,5))
    axes[0].barh(names, accs,   color='#4C72B0', alpha=0.85)
    axes[0].set_xlabel('Val Accuracy (%)'); axes[0].set_title('Do chinh xac', fontweight='bold')
    axes[1].barh(names, speeds, color='#55A868', alpha=0.85)
    axes[1].set_xlabel('Inference ms/anh (CPU)'); axes[1].set_title('Toc do', fontweight='bold')
    axes[2].scatter(speeds, accs, s=[p*15 for p in params],
                    c=range(len(names)), cmap='tab10', alpha=0.85)
    for i,n in enumerate(names):
        axes[2].annotate(n,(speeds[i],accs[i]),fontsize=8,xytext=(5,3),textcoords='offset points')
    axes[2].set_xlabel('Inference ms'); axes[2].set_ylabel('Accuracy (%)')
    axes[2].set_title('Speed vs Accuracy (bong bong = so params)', fontweight='bold')
    axes[2].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR/'model_comparison.png', dpi=120)
    plt.show()

# plot_benchmark(bench)